# ML4HL Responsible AI Toolbox: Hepatitis C Liver Disease Risk Stratification

**Course**: AI for Healthcare (AI4HC) — IE University, Sep 2026  
**Adapted from**: [IE-ML-for-Healthcare/RAI_opioid_risk_prevention](https://github.com/IE-ML-for-Healthcare/RAI_opioid_risk_prevention)  
**Dataset**: UCI Hepatitis C Dataset (615 records, 14 attributes)  

---

## Learning Objectives

1. Frame a **clinical problem** around liver disease screening using routine blood-test biomarkers  
2. Build a **reproducible, leak-proof pipeline** with scikit-learn  
3. Translate model performance into **patient impact** (false alerts, missed cases, workload)  
4. Justify an **operating point** with calibration and threshold analysis  
5. Use the **Responsible AI Dashboard** to identify deployment risks, biases, and mitigations  

### Three RAI Themes
- **Interpretability**: Which lab values drive the model's risk score?  
- **Counterfactual reasoning**: What minimal change in a patient's labs would flip their classification?  
- **Causal analysis**: Does intervening on a feature (e.g., reducing GGT) plausibly reduce disease risk?

---
## Step 0 — Environment & Imports

In [ ]:
# ============================================================
# 0. Environment & imports
# ============================================================
import warnings, os, sys
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.calibration import CalibratedClassifierCV, calibration_curve
from sklearn.metrics import (
    classification_report, confusion_matrix, brier_score_loss,
    roc_auc_score, average_precision_score, RocCurveDisplay,
    PrecisionRecallDisplay, ConfusionMatrixDisplay
)
from sklearn.dummy import DummyClassifier

# Our utility belt (same as reference repo)
from utils import (
    positive_scores, auc_report, tradeoff_table, wilson_interval,
    pick_threshold_cost, pick_threshold_recall_floor, pick_threshold_workload,
    summary_at_threshold,
    plot_recall_floor_curves, plot_cumulative_recall_at_threshold,
    plot_topk_at_threshold, make_thresholded_estimator, init_rai_dependencies
)

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

print(f"Python  : {sys.version}")
print(f"NumPy   : {np.__version__}")
print(f"pandas  : {pd.__version__}")
print(f"seed    : {RANDOM_STATE}")

---
## Step 1 — Clinical Problem Framing and Study Objective

### The Clinical Question

**Can routine blood-test biomarkers reliably flag patients who have progressed beyond healthy liver status — specifically to Hepatitis, Fibrosis, or Cirrhosis — so that they can be prioritised for confirmatory diagnostic workup?**

### Context & Motivation

Hepatitis C virus (HCV) infection is a leading cause of chronic liver disease worldwide. Many patients are asymptomatic until late-stage fibrosis or cirrhosis, at which point treatment options narrow and prognosis worsens. Early detection through **screening of routine laboratory panels** could:

- Identify patients who need a confirmatory HCV-RNA test or liver biopsy  
- Reduce diagnostic delay in resource-limited settings  
- Enable earlier antiviral treatment (direct-acting antivirals cure >95% of HCV cases)  

### Target Population
Adults presenting for blood donation or routine lab work.

### Decision Context
A **screening flag** (not a diagnosis) triggers a follow-up workup. The model assists clinicians — it does not replace confirmatory testing.

### Success Criteria
- **Recall ≥ 60%**: catch the majority of true liver-disease cases (a missed cirrhosis patient is a serious harm)  
- **Alerts ≤ 300 per 1,000 screened**: keep the workload manageable for follow-up clinics  
- **Calibrated probabilities**: the model's risk scores should be trustworthy for clinical conversations  

### Potential Harms
| Error | Clinical consequence |
|---|---|
| **False Negative** (missed case) | Delayed diagnosis → disease progression, liver failure, transmission |
| **False Positive** (false alarm) | Unnecessary anxiety, follow-up tests, cost to patient and system |

> **So what?** In a screening context, missing a true case (FN) is generally more harmful than an extra follow-up test (FP), so we prioritise **recall** while keeping alert volume manageable.

---
## Step 2 — Data Characterisation and Representativeness

In [ ]:
# ============================================================
# 2a. Load and inspect
# ============================================================
raw = pd.read_csv("Data/HepatitisCdata.csv")
print(f"Raw shape: {raw.shape}")
raw.head()

In [ ]:
# ============================================================
# 2b. Schema alignment & target engineering
# ============================================================
# Drop the unnamed index column
df = raw.drop(columns=["Unnamed: 0"], errors="ignore").copy()

# --- Binary target: Liver Disease (1) vs Healthy Donor (0) ---
# Categories: 0=Blood Donor, 0s=suspect Blood Donor → 0 (healthy)
#             1=Hepatitis, 2=Fibrosis, 3=Cirrhosis → 1 (liver disease)
disease_map = {
    "0=Blood Donor": 0,
    "0s=suspect Blood Donor": 0,  # conservative: suspects grouped with healthy
    "1=Hepatitis": 1,
    "2=Fibrosis": 1,
    "3=Cirrhosis": 1,
}
df["LiverDisease"] = df["Category"].map(disease_map)

# Keep the original category for subgroup analysis later
df["Category_Original"] = df["Category"]

# Encode Sex as binary
df["Sex_binary"] = (df["Sex"] == "m").astype(int)  # 1 = male, 0 = female

print("Target distribution:")
print(df["LiverDisease"].value_counts())
print(f"\nDisease prevalence: {df['LiverDisease'].mean():.3f}")
print(f"\nOriginal categories:")
print(df["Category_Original"].value_counts())

In [ ]:
# ============================================================
# 2c. Data dictionary
# ============================================================
data_dict = pd.DataFrame({
    "Feature": ["Age", "Sex", "ALB", "ALP", "ALT", "AST", "BIL", "CHE", "CHOL", "CREA", "GGT", "PROT"],
    "Description": [
        "Patient age (years)",
        "Biological sex (m/f)",
        "Albumin (g/L) — low in liver disease",
        "Alkaline phosphatase (IU/L) — elevated in cholestasis",
        "Alanine aminotransferase (IU/L) — hepatocyte damage marker",
        "Aspartate aminotransferase (IU/L) — hepatocyte/muscle damage",
        "Bilirubin (µmol/L) — elevated in liver dysfunction",
        "Cholinesterase (kU/L) — low in severe liver disease",
        "Cholesterol (mmol/L) — may decrease in cirrhosis",
        "Creatinine (µmol/L) — renal function / hepatorenal syndrome",
        "Gamma-glutamyl transferase (IU/L) — elevated in liver/biliary disease",
        "Total protein (g/L) — reflects liver synthetic function"
    ],
    "Type": ["numeric"] * 12,
    "Clinical_relevance": [
        "Older age → higher HCV prevalence",
        "Sex differences in disease progression",
        "↓ Albumin = poor liver synthetic function",
        "↑ ALP = cholestatic pattern",
        "↑ ALT = active hepatocellular injury",
        "↑ AST = hepatocellular injury (AST/ALT ratio > 1 suggests cirrhosis)",
        "↑ Bilirubin = impaired conjugation/excretion",
        "↓ CHE = reduced synthetic capacity",
        "↓ Cholesterol = advanced liver disease",
        "↑ Creatinine = hepatorenal involvement",
        "↑ GGT = biliary damage, alcohol use, enzyme induction",
        "↓ Total protein = synthetic failure"
    ]
})
data_dict

In [ ]:
# ============================================================
# 2d. Missingness assessment
# ============================================================
missing = df.isnull().sum()
missing_pct = (missing / len(df) * 100).round(2)
missing_df = pd.DataFrame({"missing_count": missing, "missing_pct": missing_pct})
print("Missing values:")
print(missing_df[missing_df["missing_count"] > 0])

print(f"\n→ So what? ALP has {missing_df.loc['ALP','missing_count']:.0f} missing values ({missing_df.loc['ALP','missing_pct']:.1f}%).")
print("  Median imputation inside the pipeline will handle this without leaking test info.")

In [ ]:
# ============================================================
# 2e. Cohort table & class imbalance
# ============================================================
numeric_cols = ["Age", "ALB", "ALP", "ALT", "AST", "BIL", "CHE", "CHOL", "CREA", "GGT", "PROT"]

cohort = df.groupby("LiverDisease")[numeric_cols].agg(["mean", "std", "count"]).T
print("Cohort summary (Disease=1 vs Healthy=0):")
display(cohort) if hasattr(__builtins__, '__IPYTHON__') else print(cohort)

print(f"\nSex distribution by target:")
print(pd.crosstab(df["Sex"], df["LiverDisease"], margins=True))

print(f"\n→ So what? Disease prevalence is {df['LiverDisease'].mean():.1%} — moderate imbalance.")
print("  PR AUC and recall are more informative than accuracy in this setting.")

In [ ]:
# ============================================================
# 2f. Exploratory visualisations
# ============================================================
fig, axes = plt.subplots(3, 4, figsize=(16, 10))
for ax, col in zip(axes.ravel(), numeric_cols):
    df.boxplot(column=col, by="LiverDisease", ax=ax)
    ax.set_title(col)
    ax.set_xlabel("")
# Hide unused subplot
if len(numeric_cols) < 12:
    axes.ravel()[-1].set_visible(False)
plt.suptitle("Feature distributions by Liver Disease status (0=Healthy, 1=Disease)", y=1.02)
plt.tight_layout()
plt.show()

print("→ So what? ALT, AST, GGT, BIL are visibly elevated in disease cases.")
print("  ALB and CHE are lower. These align with known liver disease biomarkers.")

In [ ]:
# ============================================================
# 2g. Correlation heatmap
# ============================================================
plt.figure(figsize=(10, 8))
corr_cols = numeric_cols + ["LiverDisease"]
sns.heatmap(df[corr_cols].corr(), annot=True, fmt=".2f", cmap="RdBu_r", center=0)
plt.title("Feature correlation matrix (including target)")
plt.tight_layout()
plt.show()

print("→ So what? Check for highly correlated features (potential multicollinearity)")
print("  and features with strong target correlation (candidates for high importance).")

### Representativeness & External Validity

**Caveats:**
- This dataset is from a **single centre** (UCI repository, originally from a blood-donation screening context)  
- The disease cohort is small (n≈75) — any subgroup analysis will have wide confidence intervals  
- No temporal information is available — we cannot verify whether labs preceded diagnosis  
- The "suspect blood donor" category is ambiguous — we conservatively group them as healthy  

> **So what?** Results should be interpreted as a **teaching exercise**, not as a validated clinical screening tool. Generalization to other populations requires external validation.

---
## Step 3 — Experimental Design and Splits

In [ ]:
# ============================================================
# 3. Train / Validation / Test split (70 / 15 / 15)
# ============================================================

# Define features and target
FEATURE_COLS = ["Age", "Sex_binary", "ALB", "ALP", "ALT", "AST", "BIL", "CHE", "CHOL", "CREA", "GGT", "PROT"]
TARGET = "LiverDisease"
SENSITIVE_FEATURE = "Sex_binary"  # for fairness analysis

X = df[FEATURE_COLS].copy()
y = df[TARGET].copy()

# First split: 70% train, 30% temp
X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.30, random_state=RANDOM_STATE, stratify=y
)

# Second split: 50/50 of temp → 15% val, 15% test
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.50, random_state=RANDOM_STATE, stratify=y_temp
)

print(f"Training  : {X_train.shape[0]} rows, {y_train.sum()} disease cases ({y_train.mean():.3f} prevalence)")
print(f"Validation: {X_val.shape[0]} rows, {y_val.sum()} disease cases ({y_val.mean():.3f} prevalence)")
print(f"Test      : {X_test.shape[0]} rows, {y_test.sum()} disease cases ({y_test.mean():.3f} prevalence)")

print("\n→ So what? Stratified split preserves disease prevalence across all sets.")
print("  Validation is used for calibration & threshold tuning. Test is locked until Step 8.")

### Design Rationale

- **Stratified split** ensures disease prevalence is preserved in each partition  
- **No temporal split** is possible (dataset has no timestamps) — a limitation we acknowledge  
- In a real deployment, a **temporal or site-based split** would better simulate prospective screening  
- `RANDOM_STATE = 42` is fixed throughout for reproducibility  
- The `environment.yml` captures all dependency versions

---
## Step 4 — Pipelines, Baselines, and Leakage Controls

In [ ]:
# ============================================================
# 4a. Define column groups
# ============================================================
NUMERIC_FEATURES = ["Age", "ALB", "ALP", "ALT", "AST", "BIL", "CHE", "CHOL", "CREA", "GGT", "PROT"]
BINARY_FEATURES  = ["Sex_binary"]

print(f"Numeric features ({len(NUMERIC_FEATURES)}): {NUMERIC_FEATURES}")
print(f"Binary features  ({len(BINARY_FEATURES)}):  {BINARY_FEATURES}")

In [ ]:
# ============================================================
# 4b. Preprocessing pipeline (ALL stateful steps inside)
# ============================================================
numeric_transformer = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
])

binary_transformer = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    # No scaling needed for 0/1 flags
])

preprocessor = ColumnTransformer([
    ("num", numeric_transformer, NUMERIC_FEATURES),
    ("bin", binary_transformer, BINARY_FEATURES),
], remainder="drop")

print("✓ All stateful transformations (imputation, scaling) are inside the pipeline.")
print("  Golden rule: SPLIT FIRST, TRANSFORM LATER — no data leakage.")

In [ ]:
# ============================================================
# 4c. Majority-class baseline
# ============================================================
baseline = DummyClassifier(strategy="most_frequent", random_state=RANDOM_STATE)
baseline.fit(X_train, y_train)

print("Baseline (always predict majority class):")
print(classification_report(y_val, baseline.predict(X_val), target_names=["Healthy", "Disease"]))

print("→ So what? The baseline has 0% recall for disease — it never flags anyone.")
print("  Any useful model must substantially beat this.")

In [ ]:
# ============================================================
# 4d. Logistic Regression pipeline (unweighted — transparent model)
# ============================================================
lr_pipeline = Pipeline([
    ("preprocessor", preprocessor),
    ("classifier", LogisticRegression(
        solver="lbfgs",
        max_iter=1000,
        random_state=RANDOM_STATE,
        # No class_weight — we handle threshold separately
    ))
])

lr_pipeline.fit(X_train, y_train)
print("✓ Logistic Regression pipeline fitted on training data only.")

In [ ]:
# ============================================================
# 4e. Coefficient interpretation
# ============================================================
feature_names = NUMERIC_FEATURES + BINARY_FEATURES
coefs = lr_pipeline.named_steps["classifier"].coef_[0]
intercept = lr_pipeline.named_steps["classifier"].intercept_[0]

coef_df = pd.DataFrame({
    "Feature": feature_names,
    "Coefficient": coefs,
    "Odds_Ratio": np.exp(coefs)
}).sort_values("Coefficient", ascending=False)

print(f"Intercept: {intercept:.4f}")
print("\nLogistic Regression coefficients (standardised features):")
display(coef_df) if hasattr(__builtins__, '__IPYTHON__') else print(coef_df)

# Plot
plt.figure(figsize=(8, 5))
plt.barh(coef_df["Feature"], coef_df["Coefficient"])
plt.xlabel("Coefficient (standardised)")
plt.title("Feature coefficients — Logistic Regression")
plt.axvline(0, color="black", linewidth=0.5)
plt.tight_layout()
plt.show()

print("\n→ So what? Coefficients describe MODEL ASSOCIATIONS, not causal effects.")
print("  Positive coefs (e.g., AST, GGT, BIL) push the risk score UP.")
print("  Negative coefs (e.g., ALB, CHE) push the risk score DOWN.")
print("  This aligns with clinical knowledge of liver function markers.")

### Leakage Controls Checklist

| Check | Status |
|---|---|
| All imputation/scaling fitted on **train only** | ✅ Inside `Pipeline` |
| No feature derived from the target | ✅ All features are pre-existing lab values |
| Validation used only for threshold/calibration | ✅ |
| Test set locked until Step 8 | ✅ |
| Random seed fixed | ✅ `RANDOM_STATE = 42` |

---
## Step 5 — Metrics and Discrimination Analysis

In [ ]:
# ============================================================
# 5a. Validation discrimination (ROC AUC, PR AUC)
# ============================================================
y_val_scores = positive_scores(lr_pipeline, X_val)

val_metrics = auc_report(y_val, y_val_scores, name="Logistic Regression (Validation)", plot=True)

In [ ]:
# ============================================================
# 5b. Subgroup analysis by sex
# ============================================================
for sex_label, sex_val in [("Male", 1), ("Female", 0)]:
    mask = X_val["Sex_binary"] == sex_val
    if mask.sum() == 0 or y_val[mask].sum() == 0:
        print(f"  {sex_label}: insufficient data for AUC")
        continue
    sub_roc = roc_auc_score(y_val[mask], y_val_scores[mask.values])
    sub_pr  = average_precision_score(y_val[mask], y_val_scores[mask.values])
    n_pos = y_val[mask].sum()
    print(f"  {sex_label}: ROC AUC = {sub_roc:.3f}, PR AUC = {sub_pr:.3f}, n_disease = {n_pos}, n_total = {mask.sum()}")

print("\n→ So what? If AUC differs substantially between sexes, the model may be")
print("  less reliable for one group — a fairness concern to investigate in Step 9.")

In [ ]:
# ============================================================
# 5c. Trade-off: false positives vs false negatives in clinical terms
# ============================================================
print("In this screening scenario:")
print("  • A FALSE POSITIVE means a healthy donor is recalled for further testing")
print("    → Cost: anxiety + unnecessary follow-up (e.g., HCV-RNA test, ~€30-100)")
print("  • A FALSE NEGATIVE means a liver disease patient is missed")
print("    → Cost: delayed diagnosis → disease progression, possible liver failure")
print("\n  In screening, FN cost >> FP cost → we lean toward higher recall.")

---
## Step 6 — Probability Calibration and Uncertainty

In [ ]:
# ============================================================
# 6a. Calibrate using CalibratedClassifierCV on TRAINING data only
# ============================================================
calibrated_clf = CalibratedClassifierCV(
    estimator=lr_pipeline,
    method="sigmoid",   # Platt scaling
    cv=5,               # 5-fold CV within training set
)
calibrated_clf.fit(X_train, y_train)

y_val_cal_scores = positive_scores(calibrated_clf, X_val)

print("✓ Calibration fitted on training folds only — no validation/test leakage.")

In [ ]:
# ============================================================
# 6b. Reliability plot + Brier score
# ============================================================
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, scores, label in [
    (axes[0], y_val_scores, "Raw LR"),
    (axes[1], y_val_cal_scores, "Calibrated LR")
]:
    fraction_pos, mean_pred = calibration_curve(y_val, scores, n_bins=8, strategy="quantile")
    brier = brier_score_loss(y_val, scores)
    ax.plot(mean_pred, fraction_pos, "s-", label=f"{label} (Brier={brier:.4f})")
    ax.plot([0, 1], [0, 1], "k--", label="Perfect calibration")
    ax.set_xlabel("Mean predicted probability")
    ax.set_ylabel("Observed fraction positive")
    ax.set_title(f"Reliability plot — {label}")
    ax.legend()

plt.tight_layout()
plt.show()

brier_raw = brier_score_loss(y_val, y_val_scores)
brier_cal = brier_score_loss(y_val, y_val_cal_scores)
print(f"Brier score (raw):        {brier_raw:.4f}")
print(f"Brier score (calibrated): {brier_cal:.4f}")

In [ ]:
# ============================================================
# 6c. Calibration-in-the-large
# ============================================================
obs_prev   = y_val.mean()
mean_raw   = y_val_scores.mean()
mean_cal   = y_val_cal_scores.mean()

print(f"Observed prevalence (validation):     {obs_prev:.3f}")
print(f"Mean raw predicted probability:       {mean_raw:.3f}")
print(f"Mean calibrated predicted probability:{mean_cal:.3f}")
print(f"Absolute difference (calibrated):     {abs(obs_prev - mean_cal):.3f}")

print("\n→ So what? Calibration-in-the-large checks if the model's average predicted")
print("  risk matches reality. A close match means the model isn't systematically")
print("  over- or under-estimating disease risk in this population.")
print("  IMPORTANT: This does NOT guarantee individual predictions are correct.")

---
## Step 7 — Threshold Selection and Decision Analysis

In [ ]:
# ============================================================
# 7a. Method 1: Recall-floor policy
# ============================================================
RECALL_FLOOR = 0.60

res_recall = pick_threshold_recall_floor(y_val, y_val_cal_scores, recall_floor=RECALL_FLOOR)
print("Policy: Recall ≥ 60%, then maximize precision")
print(res_recall["summary"].to_string(index=False))
print(f"\nChosen threshold (recall-floor): {res_recall['threshold']:.6f}")

In [ ]:
# ============================================================
# 7b. Method 2: Workload-constrained policy
# ============================================================
ALERTS_BUDGET = 300.0  # max 300 alerts per 1,000 patients screened

res_workload = pick_threshold_workload(y_val, y_val_cal_scores, alerts_per_1000_max=ALERTS_BUDGET)
print(f"Policy: ≤ {ALERTS_BUDGET:.0f} alerts per 1,000, then maximize true positives")
print(res_workload["summary"].to_string(index=False))
print(f"\nChosen threshold (workload): {res_workload['threshold']:.6f}")

In [ ]:
# ============================================================
# 7c. Method 3: Cost-sensitive analysis (illustrative)
# ============================================================
# Illustrative harm weights: missing a disease case is 10× worse than a false alarm
C_FP = 1   # cost of false positive (unnecessary follow-up)
C_FN = 10  # cost of false negative (missed liver disease)

res_cost = pick_threshold_cost(y_val, y_val_cal_scores, C_FP=C_FP, C_FN=C_FN)
print(f"Policy: Minimize expected cost (C_FP={C_FP}, C_FN={C_FN})")
print(res_cost["summary"].to_string(index=False))

In [ ]:
# ============================================================
# 7d. Joint policy: apply BOTH recall floor AND workload cap
# ============================================================
tbl = tradeoff_table(y_val, y_val_cal_scores)
feasible = tbl[(tbl["recall"] >= RECALL_FLOOR) & (tbl["alerts_per_1000"] <= ALERTS_BUDGET + 1e-9)]

if len(feasible) > 0:
    best_idx = feasible["precision"].idxmax()
    CHOSEN_THRESHOLD = float(feasible.loc[best_idx, "threshold"])
    joint_rule = "Joint: recall ≥ 60% AND alerts ≤ 300/1000, max precision"
else:
    # Fallback: use recall-floor only
    CHOSEN_THRESHOLD = res_recall["threshold"]
    joint_rule = "Fallback: recall-floor only (workload cap not feasible)"

chosen_summary = summary_at_threshold(y_val, y_val_cal_scores, CHOSEN_THRESHOLD)
print(f"LOCKED THRESHOLD: {CHOSEN_THRESHOLD:.6f}")
print(f"Rule: {joint_rule}")
print(chosen_summary.to_string(index=False))

# Wilson CI on recall
tp_val = int(chosen_summary["TP"].iloc[0])
fn_val = int(chosen_summary["FN"].iloc[0])
lo, hi = wilson_interval(tp_val, tp_val + fn_val)
print(f"\nRecall 95% Wilson CI: ({lo:.3f}, {hi:.3f})")

print("\n→ So what? This threshold is now FROZEN. We will not change it based on test results.")

In [ ]:
# ============================================================
# 7e. Visualisations at chosen threshold
# ============================================================
plot_recall_floor_curves(y_val, y_val_cal_scores, recall_floor=RECALL_FLOOR, chosen_threshold=CHOSEN_THRESHOLD)
plot_cumulative_recall_at_threshold(y_val, y_val_cal_scores, chosen_threshold=CHOSEN_THRESHOLD)
plot_topk_at_threshold(y_val, y_val_cal_scores, chosen_threshold=CHOSEN_THRESHOLD, top_k=30)

In [ ]:
# ============================================================
# 7f. Impact table: compare the three methods
# ============================================================
impact_rows = []
for method, thr in [
    ("Recall-floor", res_recall["threshold"]),
    ("Workload-cap", res_workload["threshold"]),
    ("Cost-sensitive (empirical)", res_cost["threshold_empirical"]),
    ("Joint (CHOSEN)", CHOSEN_THRESHOLD),
]:
    s = summary_at_threshold(y_val, y_val_cal_scores, thr).iloc[0].to_dict()
    s["method"] = method
    impact_rows.append(s)

impact_df = pd.DataFrame(impact_rows)[["method", "threshold", "recall", "precision", "TP", "FP", "FN", "alerts_per_1000"]]
print("Threshold comparison (validation):")
display(impact_df) if hasattr(__builtins__, '__IPYTHON__') else print(impact_df)

print("\n→ So what? Each method trades off differently. The CHOSEN threshold balances")
print("  catching most disease cases while keeping alert volume manageable for clinicians.")

---
## Step 8 — Final Test Evaluation and Generalisation

In [ ]:
# ============================================================
# 8a. Locked evaluation on held-out test set
# ============================================================
print("="*60)
print("FINAL TEST EVALUATION — threshold locked BEFORE seeing test data")
print(f"Locked threshold: {CHOSEN_THRESHOLD:.6f}")
print("="*60)

y_test_scores = positive_scores(calibrated_clf, X_test)
y_test_pred = (y_test_scores >= CHOSEN_THRESHOLD).astype(int)

# Classification report
print("\nClassification Report (Test):")
print(classification_report(y_test, y_test_pred, target_names=["Healthy", "Disease"]))

# Detailed metrics
test_summary = summary_at_threshold(y_test, y_test_scores, CHOSEN_THRESHOLD)
print("Test summary:")
print(test_summary.to_string(index=False))

# AUC on test
test_metrics = auc_report(y_test, y_test_scores, name="Calibrated LR (Test)", plot=True)

In [ ]:
# ============================================================
# 8b. Confidence intervals on test recall
# ============================================================
tp_test = int(test_summary["TP"].iloc[0])
fn_test = int(test_summary["FN"].iloc[0])
lo_test, hi_test = wilson_interval(tp_test, tp_test + fn_test)
print(f"Test recall: {test_summary['recall'].iloc[0]:.3f} (95% Wilson CI: {lo_test:.3f} – {hi_test:.3f})")

# Precision CI
fp_test = int(test_summary["FP"].iloc[0])
lo_p, hi_p = wilson_interval(tp_test, tp_test + fp_test)
print(f"Test precision: {test_summary['precision'].iloc[0]:.3f} (95% Wilson CI: {lo_p:.3f} – {hi_p:.3f})")

In [ ]:
# ============================================================
# 8c. Confusion matrix (test)
# ============================================================
ConfusionMatrixDisplay.from_predictions(
    y_test, y_test_pred,
    display_labels=["Healthy", "Disease"],
    cmap="Blues"
)
plt.title(f"Test confusion matrix (threshold = {CHOSEN_THRESHOLD:.4f})")
plt.show()

In [ ]:
# ============================================================
# 8d. Subgroup fairness check on test (by sex)
# ============================================================
print("Subgroup performance on TEST set:")
for sex_label, sex_val in [("Male", 1), ("Female", 0)]:
    mask = X_test["Sex_binary"] == sex_val
    n_total = mask.sum()
    n_disease = y_test[mask].sum()
    if n_disease == 0 or n_total == 0:
        print(f"  {sex_label}: no disease cases in test — skip")
        continue
    sub_recall = recall_score(y_test[mask], y_test_pred[mask.values])
    sub_precision = precision_score(y_test[mask], y_test_pred[mask.values], zero_division=0)
    print(f"  {sex_label} (n={n_total}, disease={n_disease}): recall={sub_recall:.3f}, precision={sub_precision:.3f}")

from sklearn.metrics import recall_score as sk_recall, precision_score as sk_precision

print("\n→ So what? If recall differs substantially between sexes, the model may")
print("  under-detect disease in one group — a critical fairness issue in screening.")
print("  Mitigation: subgroup-specific thresholds or additional features.")

### Generalisation Notes

- The test set was **never used** for model selection, calibration, or threshold tuning  
- Wide confidence intervals are expected given the small disease-positive sample  
- External validation on an independent cohort would be needed before clinical deployment  
- **Key limitation**: no temporal ordering → this is retrospective risk stratification, not prospective prediction

---
## Step 9 — Responsible AI Analysis (RAI Toolbox)

This section builds the `RAIInsights` dashboard for interactive exploration.  
Requires `responsibleai` and `raiwidgets` — install via the `environment.yml`.

In [ ]:
# ============================================================
# 9a. Initialise RAI dependencies
# ============================================================
rai_status, rai_objects = init_rai_dependencies()
globals().update(rai_objects)

print("RAI availability:")
for k, v in rai_status.items():
    print(f"  {k}: {'✅' if v else '❌'}")

In [ ]:
# ============================================================
# 9b. Prepare data for RAI dashboard
# ============================================================
# The RAI dashboard needs a single DataFrame with features + target
# and a model that accepts raw feature DataFrames

# Wrap the calibrated model with our chosen threshold
thresholded_model = make_thresholded_estimator(
    calibrated_clf,
    threshold=CHOSEN_THRESHOLD,
    positive_label=1
)
# "Fit" just to set internal attributes (model is already trained)
thresholded_model.fit(X_train, y_train)

# Prepare DataFrames with target column attached
train_df = X_train.copy()
train_df[TARGET] = y_train.values

test_df = X_test.copy()
test_df[TARGET] = y_test.values

# Identify categorical features for RAI
categorical_features = [c for c in FEATURE_COLS if X_train[c].nunique() <= 10]
print(f"Categorical features for RAI: {categorical_features}")
print(f"Train shape: {train_df.shape}, Test shape: {test_df.shape}")

In [ ]:
# ============================================================
# 9c. Build RAIInsights (if available)
# ============================================================
if rai_status["_RAI"]:
    rai_insights = RAIInsights(
        model=thresholded_model,
        train=train_df,
        test=test_df,
        target_column=TARGET,
        task_type="classification",
        categorical_features=categorical_features,
    )

    # Add components
    rai_insights.explainer.add()
    rai_insights.error_analysis.add()
    rai_insights.counterfactual.add(total_CFs=5, desired_class="opposite")
    rai_insights.causal.add(treatment_features=["ALT", "AST", "GGT", "BIL"])

    print("Computing RAI insights (this may take a few minutes)...")
    rai_insights.compute()
    print("✓ RAI insights computed successfully.")
else:
    print("⚠ RAI packages not available. Install via environment.yml to use the dashboard.")
    print("  You can still analyse the model using the metrics and plots above.")

In [ ]:
# ============================================================
# 9d. Launch the Responsible AI Dashboard
# ============================================================
if rai_status["_RAI"]:
    ResponsibleAIDashboard(rai_insights)
else:
    print("Dashboard not available — see installation instructions in README.md")

### RAI Dashboard Exploration Guide

Once the dashboard loads, explore these components:

#### 1. Data Analysis
- Examine feature distributions for the disease vs healthy groups  
- **Form a cohort**: e.g., `Sex_binary == 1` (males) — do they have different risk profiles?  

#### 2. Model Overview & Fairness
- Check overall accuracy, recall, precision at the chosen threshold  
- **Fairness**: compare recall/FPR across `Sex_binary` — is the model equitable?  
- **Action**: if recall differs by >10pp between sexes, consider subgroup thresholds  

#### 3. Error Analysis
- The error tree highlights patient segments with high error rates  
- **Look for**: does the model struggle with borderline fibrosis cases? Young patients?  
- **Action**: targeted data collection or separate models for high-error cohorts  

#### 4. Feature Importance
- Global: which features drive the model most? (Expect: AST, ALT, GGT, BIL, ALB, CHE)  
- Local: for a specific patient, which labs pushed their score up/down?  
- **Action**: validate that top features align with clinical liver-disease knowledge  

#### 5. Counterfactuals
- "What would need to change for this patient to be classified differently?"  
- **Example**: a flagged patient might flip to healthy if GGT drops by 50 IU/L  
- **Action**: use for clinician-ready explanations ("the model flagged you primarily because of elevated GGT and AST")  

#### 6. Causal Analysis
- Estimate treatment effects: "If we could reduce ALT by X, what happens to the outcome?"  
- **Caveat**: causal claims require strong assumptions — these are exploratory, not confirmatory  
- **Action**: identify candidate interventions for prospective clinical studies

### Non-trivial Insights to Look For

1. **AST/ALT ratio**: cirrhosis patients often have AST > ALT (De Ritis ratio > 1). Does the model implicitly learn this?  
2. **CHE as a severity marker**: cholinesterase drops in advanced disease. Is it driving errors in mild hepatitis?  
3. **Sex-based disparities**: if the model's recall is lower for females, it may reflect training-data imbalance or genuine biological differences in biomarker expression  
4. **Creatinine outliers**: very high creatinine may indicate hepatorenal syndrome — does the error tree flag these?  

### Proposed Mitigations

| Risk | Mitigation | Monitoring |
|---|---|---|
| Low recall in females | Subgroup-specific thresholds (Fairlearn) | Track recall by sex monthly |
| Missed mild hepatitis | Add HCV-specific biomarkers (e.g., viral load) | Alert when FN rate exceeds 40% in any cohort |
| Calibration drift | Re-calibrate quarterly on recent data | Compare predicted vs observed prevalence |
| Overreliance on GGT | GGT is also elevated by alcohol — add alcohol history feature | Feature importance monitoring |

---
## Summary & Policy Implications

### What We Built
A transparent logistic regression model that uses **12 routine blood-test features** to flag patients at elevated risk of liver disease (hepatitis, fibrosis, or cirrhosis) in a screening setting.

### Key Results
- **Discrimination**: the model meaningfully separates disease from healthy cases (check ROC AUC and PR AUC above)  
- **Calibration**: predicted probabilities approximately match observed disease rates  
- **Threshold**: chosen to prioritise recall while keeping alert volume manageable  

### Policy Implications

| Aspect | Pro | Con |
|---|---|---|
| **Screening adoption** | Cheap, uses existing lab data | Requires confirmatory testing for all alerts |
| **Resource allocation** | Prioritises follow-up for highest-risk patients | Alert fatigue if false-positive rate is too high |
| **Equity** | Can flag underserved patients who miss regular screening | May underperform for demographic groups underrepresented in training |
| **Transparency** | Logistic regression is fully interpretable | May sacrifice some accuracy vs. black-box models |

### Next Steps for Deployment
1. **External validation** on an independent cohort (different hospital/geography)  
2. **Prospective pilot** with clinician feedback on alert utility  
3. **Monitoring plan**: track recall, precision, and calibration monthly; alert if recall drops below 50% in any subgroup  
4. **Governance**: institutional review board approval, patient consent for screening flag  